In [1]:
# PHASE 2 - ABLATION 6 (Lunit ViT-S/16): LABEL EFFICIENCY of the frozen features  (original | SimCLR self | adjacency | CCSM ms | CCSM fms | combined), seeds 42/43/44
# Kaggle: GPU on, Internet on.  Checkpoints are found automatically:
#   * adjacency / self (adjclv4-lunit_vit_small_{neighbor,self}_s<seed>_best.pt)  -> dataset sethu2954/adjc-lunit-checkpoints  (downloaded via kagglehub if not attached)
#   * CCSM ms / fms    (ccsm_lunit_{ms,fms}_s<seed>_best.pt)                      -> dataset badrinathachanta/ccsm-checkpoints-2  (downloaded via kagglehub if not attached)
#   * combined         (abl5-comb_vit_small_neighbor_s<seed>_best.pt)             -> dataset badrinathachanta/ablation-combined-checkpoints (ATTACH it as an input; skipped with a message if absent)
# Protocol: probe trained on a random fraction {1, 5, 10, 25, 100}% of the labelled patches of centres 0,1,3,4 (5/5/3/2/1 random draws), tested on the full unseen centre 2 (AUROC).
# Same probe as before (StandardScaler + LogisticRegression C=1 balanced, max_iter 2000).  Estimated runtime ~7-9 h (16 models: extraction ~20-25 min each + probes ~8 min each);
# resumable: a small .json is saved per model and the feature file is deleted; it stops cleanly before BUDGET_H=11 h -> Save Version and re-run with that OUTPUT attached as an input.
!pip install -q timm datasets kagglehub transformers

"""
Label-efficiency analysis.  Caveat: the adjacency / CCSM models saw the UNLABELED Camelyon17 images of centres 0,1,3,4 during pretraining (centre 2 never), so centre 2 is a clean test.
"""
import os, gc, json, glob, random, time, shutil, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score
from tqdm import tqdm

warnings.filterwarnings('ignore')
T_START = time.time()


class Config:
    RUN_FAMILIES = ['lunit']
    SEEDS = [42, 43, 44]
    BUDGET_H = 11.0                       # stop starting new chunks when the next one would not fit
    HF_DATASET = "wltjr1007/Camelyon17-WILDS"
    CKPT_SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working/checkpoints"]
    KAGGLE_CKPT_DATASETS = ["sethu2954/adjc-lunit-checkpoints", "badrinathachanta/ccsm-checkpoints-2"]   # fallback if not attached
    PRIOR_RESULT_ROOTS = ["/kaggle/input"]                                  # earlier runs' outputs (p2_eval_all/*.json) are reused
    FEATURE_DIR = "/kaggle/working/p2_features_label"
    RESULTS_DIR = "/kaggle/working/p2_eval_label"
    KEEP_FEATURES = False

    EXTRACT_BATCH = 128
    NUM_WORKERS = 4
    ALL_CENTERS = [0, 1, 2, 3, 4]
    TEST_CENTER = 2
    LR_C, LR_MAX_ITER, PROBE_SEED = 1.0, 2000, 42
    BOOT_N, BOOT_SEED = 1000, 7
    LOCO_FOLDS = [0, 1, 2, 3, 4]
    CID_TRAIN_PER_CENTER, CID_TEST_PER_CENTER, CID_VAL_FRAC = 3000, 1000, 0.25

    FAMILIES = {   # kind, hub id / timm name, input size, fp16 extraction, Part-I LOCO of the original (pipeline check; None = ImageNet control)
        'resnet18':  dict(kind='resnet', hub=None, size=96,  amp=False, ref=0.9528, label='ImageNet ResNet18'),
        'vit_small': dict(kind='timm', hub='vit_small_patch16_224', size=224, amp=False, ref=None, label='ImageNet ViT-S/16'),
        'swin_tiny': dict(kind='timm', hub='swin_tiny_patch4_window7_224', size=224, amp=False, ref=None, label='ImageNet Swin-T'),
        'kaiko':     dict(kind='timm', hub='hf-hub:1aurent/vit_small_patch16_224.kaiko_ai_towards_large_pathology_fms', size=224, amp=False, ref=0.9873, label='Kaiko ViT-S/16'),
        'lunit':     dict(kind='timm', hub='hf-hub:1aurent/vit_small_patch16_224.lunit_dino', size=224, amp=False, ref=0.9814, label='Lunit ViT-S/16'),
        'phikon':    dict(kind='hf', hub='owkin/phikon', size=224, amp=True, ref=0.9818, label='Phikon ViT-B/16'),
    }
    ARMS = {'lunit': ['self', 'neighbor', 'ms', 'fms', 'comb']}
    DEFAULT_ARMS = ['control', 'ms', 'fms']
    PART1_OTHERS = "Part-I LOCO references: Stain Aug 0.9884 | Kaiko 0.9873 | Phikon 0.9818 | Lunit 0.9814 | ImageNet ResNet18 0.9528"

    @staticmethod
    def arms(fam):
        return Config.ARMS.get(fam, Config.DEFAULT_ARMS)

    @staticmethod
    def chunks(fam):
        """[(chunk_models)] : chunk 1 = original + seed 42, then one chunk per further seed.  Model name = '<family>__ctrl' / '<family>__<arm>_s<seed>'."""
        out = []
        for k, s in enumerate(Config.SEEDS):
            c = [f'{fam}__ctrl'] if k == 0 else []
            c += [f'{fam}__{a}_s{s}' for a in Config.arms(fam)]
            out.append(c)
        return out


def parse(name):
    fam, rest = name.split('__')
    if rest == 'ctrl':
        return fam, 'ctrl', None
    arm, s = rest.rsplit('_s', 1)
    return fam, arm, int(s)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


# ---------------------------------------------------------------- encoders (same construction as Part I / the training notebooks)
class Encoder(nn.Module):
    def __init__(self, fam):
        super().__init__()
        spec = Config.FAMILIES[fam]; self.pool = None
        if spec['kind'] == 'resnet':
            net = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
            self.backbone = nn.Sequential(*list(net.children())[:-1]); self.pool = nn.AdaptiveAvgPool2d(1); self.dim = 512
        elif spec['kind'] == 'hf':
            from transformers import AutoModel
            class PhikonBackbone(nn.Module):
                def __init__(self):
                    super().__init__()
                    self.vit = AutoModel.from_pretrained(spec['hub'])
                def forward(self, x):
                    return self.vit(pixel_values=x).last_hidden_state[:, 0]           # CLS token, as in Part I
            self.backbone = PhikonBackbone(); self.dim = self.backbone.vit.config.hidden_size
        else:
            import timm
            self.backbone = timm.create_model(spec['hub'], pretrained=True, num_classes=0)
            self.dim = self.backbone.num_features

    @torch.no_grad()
    def encode(self, x):
        f = self.backbone(x)
        if f.dim() == 4:
            f = torch.flatten(self.pool(f), 1)
        elif f.dim() == 3:
            f = f.mean(1)
        return f


def ensure_checkpoint_roots():
    if glob.glob("/kaggle/input/**/ccsm_*_best.pt", recursive=True):
        return
    import kagglehub
    for slug in Config.KAGGLE_CKPT_DATASETS:
        try:
            path = kagglehub.dataset_download(slug); print("✓ checkpoints downloaded via kagglehub:", path)
            Config.CKPT_SEARCH_ROOTS.append(path)
        except Exception as e:
            print(f"⚠ could not download {slug}: {e}")


def find_checkpoint(fam, arm, seed):
    fname = f"ccsm_{fam}_{arm}_s{seed}_best.pt"
    for root in Config.CKPT_SEARCH_ROOTS:
        hits = glob.glob(os.path.join(root, '**', fname), recursive=True)
        if hits:
            return hits[0]
    return None


def build_model(name):
    fam, arm, seed = parse(name)
    if arm == 'ctrl':
        return Encoder(fam)
    path = find_checkpoint(fam, arm, seed)
    if path is None:
        return None
    enc = Encoder(fam)
    ck = torch.load(path, map_location='cpu')
    missing, unexpected = enc.load_state_dict(ck['model_state_dict'], strict=False)
    bad = [k for k in unexpected if not k.startswith(('head.', 'adv_head.'))]
    assert not missing and not bad, f"{name}: missing={missing[:3]} unexpected={bad[:3]}"
    print(f"  [{name}] loaded {os.path.basename(path)}  (epoch {ck.get('epoch','?')}, final loss {ck.get('best_val', float('nan')):.4f})")
    return enc


# ---------------------------------------------------------------- one-pass feature extraction
class MultiSizeDataset(Dataset):
    def __init__(self, hf, sizes):
        self.hf, self.sizes = hf, sorted(sizes)
        norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        self.tf = {s: transforms.Compose([transforms.Resize((s, s)), transforms.ToTensor(), norm]) for s in self.sizes}

    def __len__(self):
        return len(self.hf)

    def __getitem__(self, i):
        img = self.hf[int(i)]['image']
        if img.mode != 'RGB':
            img = img.convert('RGB')
        return {s: self.tf[s](img) for s in self.sizes}


def feat_path(name):
    return f"{Config.FEATURE_DIR}/{name}.npy"


def extract_missing(hf, names, device):
    todo = [n for n in names if not os.path.exists(feat_path(n))]
    if not todo:
        return
    mods = {}
    for n in todo:
        m = build_model(n)
        if m is None:
            print(f"  [{n}] checkpoint not found -> skipped")
            continue
        mods[n] = m.to(device).eval()
    if not mods:
        return
    fam = parse(todo[0])[0]; size = Config.FAMILIES[fam]['size']; amp = Config.FAMILIES[fam]['amp'] and device.type == 'cuda'
    N = len(hf)
    out = {n: np.empty((N, mods[n].dim), dtype=np.float32) for n in mods}
    loader = DataLoader(MultiSizeDataset(hf, {size}), batch_size=Config.EXTRACT_BATCH, shuffle=False,
                        num_workers=Config.NUM_WORKERS, pin_memory=device.type == 'cuda')
    pos, t0 = 0, time.time()
    for batch in tqdm(loader, desc=f"extract {fam}: {len(mods)} models"):
        x = batch[size].to(device, non_blocking=True); B = x.size(0)
        for n, m in mods.items():
            with torch.autocast('cuda', enabled=amp):
                f = m.encode(x)
            out[n][pos:pos + B] = f.float().cpu().numpy()
        pos += B
    assert pos == N, (pos, N)
    os.makedirs(Config.FEATURE_DIR, exist_ok=True)
    for n in mods:
        np.save(feat_path(n), out[n])
    print(f"✓ extracted {list(mods)} in {(time.time()-t0)/60:.1f} min")
    del mods, out; gc.collect(); torch.cuda.empty_cache()


# ---------------------------------------------------------------- evaluation (same recipe as Part I)
def bootstrap_auroc_by_patient(y, p, pat, n_boot, seed):
    pats = np.unique(pat)
    idx_by = {q: np.where(pat == q)[0] for q in pats}
    rng, vals = np.random.RandomState(seed), []
    for _ in range(n_boot):
        s = rng.choice(pats, size=len(pats), replace=True)
        idx = np.concatenate([idx_by[q] for q in s])
        if len(np.unique(y[idx])) < 2:
            continue
        vals.append(roc_auc_score(y[idx], p[idx]))
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return float(lo), float(hi)


def probe_auroc(X, y, centers, patients, train_c, test_c, with_ci=False):
    tr, te = np.isin(centers, train_c), centers == test_c
    sc = StandardScaler().fit(X[tr])
    clf = LogisticRegression(max_iter=Config.LR_MAX_ITER, C=Config.LR_C, class_weight='balanced',
                             random_state=Config.PROBE_SEED, n_jobs=-1)
    clf.fit(sc.transform(X[tr]), y[tr])
    prob = clf.predict_proba(sc.transform(X[te]))[:, 1]
    auc = roc_auc_score(y[te], prob)
    bal = balanced_accuracy_score(y[te], prob > 0.5)
    if with_ci:
        lo, hi = bootstrap_auroc_by_patient(y[te], prob, patients[te], Config.BOOT_N, Config.BOOT_SEED)
        return auc, bal, lo, hi
    return auc, bal


def center_id_probe(X, centers, patients):
    """Patch-level 5-way center-identity probe, patient-disjoint train/test within each center."""
    rng = np.random.RandomState(Config.PROBE_SEED)
    tr_idx, te_idx = [], []
    for c in Config.ALL_CENTERS:
        pats = sorted(np.unique(patients[centers == c]).tolist())
        random.Random(Config.PROBE_SEED + c).shuffle(pats)
        n_te = min(max(1, int(round(Config.CID_VAL_FRAC * len(pats)))), len(pats) - 1)
        te_p, tr_p = set(pats[:n_te]), set(pats[n_te:])
        for pool, n_want, store in ((np.where((centers == c) & np.isin(patients, list(tr_p)))[0], Config.CID_TRAIN_PER_CENTER, tr_idx),
                                    (np.where((centers == c) & np.isin(patients, list(te_p)))[0], Config.CID_TEST_PER_CENTER, te_idx)):
            store.append(rng.choice(pool, size=min(n_want, len(pool)), replace=False))
    tr_idx, te_idx = np.concatenate(tr_idx), np.concatenate(te_idx)
    sc = StandardScaler().fit(X[tr_idx])
    clf = LogisticRegression(max_iter=Config.LR_MAX_ITER, n_jobs=-1).fit(sc.transform(X[tr_idx]), centers[tr_idx])
    return 100 * accuracy_score(centers[te_idx], clf.predict(sc.transform(X[te_idx])))




def ensure_checkpoint_roots():
    need = {"sethu2954/adjc-lunit-checkpoints": "adjclv4-lunit_vit_small_neighbor_s*_best.pt", "badrinathachanta/ccsm-checkpoints-2": "ccsm_lunit_ms_s*_best.pt",
            "badrinathachanta/ablation-combined-checkpoints": "abl5-comb_vit_small_neighbor_s*_best.pt"}
    import kagglehub
    for slug, pat in need.items():
        if glob.glob(os.path.join("/kaggle/input", "**", pat), recursive=True):
            continue
        try:
            path = kagglehub.dataset_download(slug); print("✓ checkpoints downloaded via kagglehub:", path)
            Config.CKPT_SEARCH_ROOTS.append(path)
        except Exception as e:
            print(f"⚠ could not download {slug}: {e}")


def find_checkpoint(fam, arm, seed):
    fname = {'self': f"adjclv4-lunit_vit_small_self_s{seed}_best.pt", 'neighbor': f"adjclv4-lunit_vit_small_neighbor_s{seed}_best.pt",
             'ms': f"ccsm_lunit_ms_s{seed}_best.pt", 'fms': f"ccsm_lunit_fms_s{seed}_best.pt",
             'comb': f"abl5-comb_vit_small_neighbor_s{seed}_best.pt"}[arm]
    for root in Config.CKPT_SEARCH_ROOTS:
        hits = glob.glob(os.path.join(root, '**', fname), recursive=True)
        if hits:
            return hits[0]
    return None


FRACS = [0.01, 0.05, 0.10, 0.25, 1.0]
DRAWS = {0.01: 5, 0.05: 5, 0.10: 3, 0.25: 2, 1.0: 1}


def evaluate_model(name, y, centers, patients):
    cache = f"{Config.RESULTS_DIR}/{name}.json"
    if os.path.exists(cache):
        return json.load(open(cache))
    X = np.ascontiguousarray(np.load(feat_path(name), mmap_mode='r'))
    tr = np.isin(centers, [c for c in Config.ALL_CENTERS if c != Config.TEST_CENTER]); te = centers == Config.TEST_CENTER
    Xtr, ytr, Xte, yte = X[tr], y[tr], X[te], y[te]
    t0, out = time.time(), {}
    for f in FRACS:
        aucs = []
        for d in range(DRAWS[f]):
            rng = np.random.RandomState(1000 + d)
            sel = np.arange(len(ytr)) if f >= 1.0 else rng.choice(len(ytr), int(round(f * len(ytr))), replace=False)
            if len(np.unique(ytr[sel])) < 2:
                continue
            sc = StandardScaler().fit(Xtr[sel])
            clf = LogisticRegression(max_iter=Config.LR_MAX_ITER, C=Config.LR_C, class_weight='balanced', random_state=Config.PROBE_SEED, n_jobs=-1)
            clf.fit(sc.transform(Xtr[sel]), ytr[sel])
            aucs.append(float(roc_auc_score(yte, clf.predict_proba(sc.transform(Xte))[:, 1])))
        out[str(f)] = aucs
    res = {'model': name, 'fracs': out}
    print(f"  [{name}] AUROC on centre 2 by label fraction: " + " | ".join(f"{int(float(k) * 100)}%: {np.mean(v):.4f}" for k, v in out.items()) + f" | {(time.time()-t0)/60:.1f} min")
    os.makedirs(Config.RESULTS_DIR, exist_ok=True)
    json.dump(res, open(cache, 'w'))
    del X, Xtr, Xte; gc.collect()
    if not Config.KEEP_FEATURES and os.path.exists(feat_path(name)):
        os.remove(feat_path(name))
    return res


LAB = {'ctrl': 'ORIGINAL Lunit', 'self': 'SimCLR self control (adjacency recipe)', 'neighbor': 'adjacency (neighbour positives)',
       'ms': 'CCSM: MixStyle', 'fms': 'CCSM: MixStyle + Fourier', 'comb': 'COMBINED adjacency + CCSM'}


def summarize():
    res = {}
    for p in glob.glob(f"{Config.RESULTS_DIR}/*.json"):
        r = json.load(open(p)); res[r['model']] = r
    per = {}                                   # arm -> seed -> {frac: mean AUROC over draws}
    for n, r in res.items():
        f, a, s = parse(n)
        per.setdefault(a, {})[s] = {float(k): float(np.mean(v)) for k, v in r['fracs'].items() if v}
    print("\n" + "=" * 112 + "\nLUNIT ViT-S  LABEL EFFICIENCY  (AUROC on centre 2; probe trained on a fraction of centres 0,1,3,4; mean ± std over seeds)\n" + "=" * 112)
    print(f"  {'method':40s}" + "".join(f"{int(fr * 100):>4d}% labels   " for fr in FRACS))
    rows = []
    for a in ['ctrl', 'self', 'neighbor', 'ms', 'fms', 'comb']:
        d = per.get(a)
        if not d: continue
        line, row = f"  {LAB[a]:40s}", dict(method=LAB[a], seeds=','.join(map(str, sorted(s for s in d if s is not None))))
        for fr in FRACS:
            v = [x[fr] for x in d.values() if fr in x]
            m, sd = _ms(v)
            line += f"{m:.4f}" + (f"±{sd:.4f}" if len(v) > 1 else "       ") + " "
            row[f"{int(fr * 100)}%"] = f"{m:.4f}" + (f" ± {sd:.4f}" if len(v) > 1 else "")
        print(line); rows.append(row)
    print("\nPaired differences per label fraction (x100 AUROC, same seed; mean over seeds):")
    for a, b, label in [('neighbor', 'self', 'adjacency minus self control'), ('neighbor', 'ctrl', 'adjacency minus ORIGINAL'), ('fms', 'ctrl', 'CCSM (MS+Fourier) minus ORIGINAL'),
                        ('ms', 'ctrl', 'CCSM (MixStyle) minus ORIGINAL'), ('comb', 'neighbor', 'combined minus adjacency'), ('comb', 'ctrl', 'combined minus ORIGINAL')]:
        if a in per and b in per:
            line = f"  {label:36s}"
            for fr in FRACS:
                if b == 'ctrl':
                    g = [100 * (x[fr] - list(per['ctrl'].values())[0][fr]) for x in per[a].values() if fr in x]
                else:
                    common = sorted(set(per[a]) & set(per[b]))
                    g = [100 * (per[a][s][fr] - per[b][s][fr]) for s in common if fr in per[a][s] and fr in per[b][s]]
                if g:
                    line += f"{np.mean(g):+7.2f}" + f"({sum(x > 0 for x in g)}/{len(g)}) "
            print(line)
    pd.DataFrame(rows).to_csv(f"{Config.RESULTS_DIR}/label_efficiency_summary.csv", index=False)
    expected = [n for fam in Config.RUN_FAMILIES for c in Config.chunks(fam) for n in c]
    missing = [n for n in expected if not have_result(n)]
    print(f"\nProgress: {len(expected) - len(missing)}/{len(expected)} models evaluated." + (f" Still missing: {missing[:12]}{' ...' if len(missing) > 12 else ''}" if missing else " ALL DONE."))
    print("Caveat: pretrained models saw unlabeled centre 0,1,3,4 images; centre 2 (the test set here) is clean.")

def _ms(v):
    v = np.array(v, dtype=float)
    return v.mean(), (v.std(ddof=1) if len(v) > 1 else float('nan'))



def load_prior_results():
    os.makedirs(Config.RESULTS_DIR, exist_ok=True)
    n = 0
    for root in Config.PRIOR_RESULT_ROOTS:
        for p in glob.glob(os.path.join(root, '**', 'p2_eval_label', '*.json'), recursive=True):
            dst = os.path.join(Config.RESULTS_DIR, os.path.basename(p))
            if not os.path.exists(dst):
                shutil.copy2(p, dst); n += 1
    if n:
        print(f"✓ reused {n} finished results from earlier runs")


def have_result(name):
    return os.path.exists(f"{Config.RESULTS_DIR}/{name}.json")


def load_meta(hf):
    return (np.array(hf['label']).astype(np.int64), np.array(hf['center']).astype(np.int64),
            np.array(hf['patient']).astype(np.int64))



def main(hf=None):
    set_seed(Config.PROBE_SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"device: {device}")
    ensure_checkpoint_roots(); load_prior_results()
    if hf is None:
        from datasets import load_dataset, concatenate_datasets
        ds = load_dataset(Config.HF_DATASET)
        parts = [ds[k] for k in ds.keys()]
        hf = concatenate_datasets(parts) if len(parts) > 1 else parts[0]
    print(f"✓ {len(hf):,} patches")
    y, centers, patients = load_meta(hf)
    os.makedirs(Config.FEATURE_DIR, exist_ok=True)
    last_chunk_h = 0.0
    for fam in Config.RUN_FAMILIES:
        for ci, chunk in enumerate(Config.chunks(fam), 1):
            todo = [n for n in chunk if not have_result(n)]
            if not todo:
                continue
            elapsed = (time.time() - T_START) / 3600
            if elapsed + max(last_chunk_h, 0.3) > Config.BUDGET_H:
                print(f"\n⏸ time budget reached ({elapsed:.1f} h used). Save this version and continue in a new session (attach this output as input).")
                return summarize()
            print(f"\n=== {fam} chunk {ci}: {todo}")
            t0 = time.time()
            extract_missing(hf, todo, device)
            for n in todo:
                if os.path.exists(feat_path(n)):
                    evaluate_model(n, y, centers, patients)
                else:
                    print(f"  [{n}] no features (checkpoint missing) -> not evaluated")
            last_chunk_h = (time.time() - t0) / 3600
            print(f"  chunk took {last_chunk_h:.2f} h | total elapsed {(time.time() - T_START) / 3600:.1f} h")
    return summarize()


# ====================================================================
# RUN
# ====================================================================
results_df = main()


device: cuda


README.md:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

data/train-00000-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  470MB            

data/train-00000-of-00014.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  469MB            

...

✓ extracted ['lunit__ctrl', 'lunit__self_s42', 'lunit__neighbor_s42', 'lunit__ms_s42', 'lunit__fms_s42', 'lunit__comb_s42'] in 146.6 min
  [lunit__ctrl] AUROC on centre 2 by label fraction: 1%: 0.9643 | 5%: 0.9787 | 10%: 0.9831 | 25%: 0.9792 | 100%: 0.9783 | 1.8 min
  [lunit__self_s42] AUROC on centre 2 by label fraction: 1%: 0.9791 | 5%: 0.9848 | 10%: 0.9855 | 25%: 0.9855 | 100%: 0.9853 | 2.7 min
  [lunit__neighbor_s42] AUROC on centre 2 by label fraction: 1%: 0.9885 | 5%: 0.9912 | 10%: 0.9914 | 25%: 0.9913 | 100%: 0.9921 | 1.7 min
  [lunit__ms_s42] AUROC on centre 2 by label fraction: 1%: 0.9857 | 5%: 0.9891 | 10%: 0.9893 | 25%: 0.9879 | 100%: 0.9873 | 1.8 min
  [lunit__fms_s42] AUROC on centre 2 by label fraction: 1%: 0.9829 | 5%: 0.9865 | 10%: 0.9860 | 25%: 0.9862 | 100%: 0.9873 | 1.7 min
  [lunit__comb_s42] AUROC on centre 2 by label fraction: 1%: 0.9849 | 5%: 0.9895 | 10%: 0.9901 | 25%: 0.9906 | 100%: 0.9906 | 1.9 min
  chunk took 2.65 h | total elapsed 2.7 h

=== lunit chunk 2: 

extract lunit: 5 models: 100%|██████████| 3563/3563 [2:02:17<00:00,  2.06s/it]


✓ extracted ['lunit__self_s43', 'lunit__neighbor_s43', 'lunit__ms_s43', 'lunit__fms_s43', 'lunit__comb_s43'] in 122.3 min
  [lunit__self_s43] AUROC on centre 2 by label fraction: 1%: 0.9803 | 5%: 0.9851 | 10%: 0.9858 | 25%: 0.9861 | 100%: 0.9860 | 2.9 min
  [lunit__neighbor_s43] AUROC on centre 2 by label fraction: 1%: 0.9876 | 5%: 0.9910 | 10%: 0.9913 | 25%: 0.9918 | 100%: 0.9919 | 1.6 min
  [lunit__ms_s43] AUROC on centre 2 by label fraction: 1%: 0.9849 | 5%: 0.9859 | 10%: 0.9859 | 25%: 0.9840 | 100%: 0.9818 | 1.8 min
  [lunit__fms_s43] AUROC on centre 2 by label fraction: 1%: 0.9823 | 5%: 0.9866 | 10%: 0.9866 | 25%: 0.9865 | 100%: 0.9874 | 1.9 min
  [lunit__comb_s43] AUROC on centre 2 by label fraction: 1%: 0.9868 | 5%: 0.9899 | 10%: 0.9903 | 25%: 0.9901 | 100%: 0.9900 | 1.9 min
  chunk took 2.22 h | total elapsed 4.9 h

=== lunit chunk 3: ['lunit__self_s44', 'lunit__neighbor_s44', 'lunit__ms_s44', 'lunit__fms_s44', 'lunit__comb_s44']
  [lunit__self_s44] loaded adjclv4-lunit_vit_sma

extract lunit: 5 models: 100%|██████████| 3563/3563 [2:02:17<00:00,  2.06s/it]


✓ extracted ['lunit__self_s44', 'lunit__neighbor_s44', 'lunit__ms_s44', 'lunit__fms_s44', 'lunit__comb_s44'] in 122.3 min
  [lunit__self_s44] AUROC on centre 2 by label fraction: 1%: 0.9794 | 5%: 0.9850 | 10%: 0.9854 | 25%: 0.9861 | 100%: 0.9862 | 2.9 min
  [lunit__neighbor_s44] AUROC on centre 2 by label fraction: 1%: 0.9882 | 5%: 0.9912 | 10%: 0.9920 | 25%: 0.9922 | 100%: 0.9925 | 1.6 min
  [lunit__ms_s44] AUROC on centre 2 by label fraction: 1%: 0.9879 | 5%: 0.9891 | 10%: 0.9900 | 25%: 0.9888 | 100%: 0.9871 | 1.7 min
  [lunit__fms_s44] AUROC on centre 2 by label fraction: 1%: 0.9830 | 5%: 0.9868 | 10%: 0.9868 | 25%: 0.9870 | 100%: 0.9880 | 1.8 min
  [lunit__comb_s44] AUROC on centre 2 by label fraction: 1%: 0.9879 | 5%: 0.9903 | 10%: 0.9907 | 25%: 0.9913 | 100%: 0.9918 | 1.9 min
  chunk took 2.22 h | total elapsed 7.1 h

LUNIT ViT-S  LABEL EFFICIENCY  (AUROC on centre 2; probe trained on a fraction of centres 0,1,3,4; mean ± std over seeds)
  method                                  